# B cell clustering and marker summaries

This notebook keeps the downstream B-cell clustering summary intentionally small: Leiden clusters, marker dot plots, simple cluster labels, and a marker heatmap using the original labels stored in the AnnData object.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

cwd = Path.cwd()
ANALYSIS_DIR = (
    cwd
    if cwd.name == "06_b_cell_igvf"
    else Path("06_b_cell_igvf")
    if Path("06_b_cell_igvf").exists()
    else Path("..").resolve()
)
DATA_DIR = Path("data/flowmap_manuscript/b_cell")
if not DATA_DIR.exists():
    DATA_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell"
FIGURE_DIR = ANALYSIS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(DATA_DIR / "bcell_velocity_standard.h5ad")
with open(DATA_DIR / "flowmap_emb.pkl", "rb") as f:
    emb = pickle.load(f)

adata.obsm["X_flowmap"] = emb.X_emb
adata


In [ ]:
# Keep clustering simple and reproducible.
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.leiden(adata, resolution=0.6)


In [ ]:
markers = {
    "Naive": ["GRASP", "ZNF331", "IRS2", "LIX1-AS1", "NR4A2", "KCNH8", "DUSP1", "RASGEF1B", "FOSB", "ZBTB10"],
    "ActB": ["ITGA1", "JAZF1", "SMIM14", "AC083837.1", "ST6GALNAC3", "IL7", "NIBAN3"],
    "preGCBC": ["KIAA1549L", "CFI", "HOMER2", "EEPD1", "PALLD", "SLC37A3", "FCER2"],
    "prePB": ["IL2RB", "IL2RA", "CD226", "DUSP4", "ACSL4", "CLIC5", "IL12RB2"],
    "PB": ["CFAP54", "AL591518.1", "ACOXL", "FNDC3B", "AC016074.2", "NUGGC", "RASSF6", "ZNF215"],
}

markers_filtered = {
    label: [gene for gene in genes if gene in adata.var_names]
    for label, genes in markers.items()
}
markers_filtered = {label: genes for label, genes in markers_filtered.items() if genes}

sc.pl.dotplot(
    adata,
    markers_filtered,
    groupby="leiden",
    standard_scale="var",
)


In [ ]:
cluster_map = {
    "0": "prePB",
    "1": "PreGCBC",
    "2": "PB",
    "3": "ActB",
    "4": "ActB",
    "5": "ActB",
    "6": "PB",
    "7": "Naive",
    "8": "Naive",
    "9": "Naive",
    "10": "Naive",
    "11": "PreGCBC",
    "12": "Naive",
}

adata.obs["bcell_cluster_label"] = adata.obs["leiden"].astype(str).map(cluster_map).astype("category")
adata.obs[["leiden", "bcell_cluster_label"]].value_counts().sort_index()


In [ ]:
heatmap_genes = sorted({
    gene
    for genes in markers_filtered.values()
    for gene in genes
})

# Use the original labels already carried by the AnnData object. If they are absent,
# fall back to the simple cluster labels assigned above.
label_key = "celltype" if "celltype" in adata.obs else "bcell_cluster_label"
labels = adata.obs[label_key].astype("category")

expr = pd.DataFrame(index=labels.cat.categories, columns=heatmap_genes, dtype=float)
for label in labels.cat.categories:
    X = adata[labels == label, heatmap_genes].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    expr.loc[label] = np.asarray(X).mean(axis=0)

expr_z = (expr - expr.mean(axis=0)) / (expr.std(axis=0) + 1e-8)

fig_width = max(8, 0.25 * len(heatmap_genes))
fig_height = max(4, 0.45 * len(expr_z.index))

fig, ax = plt.subplots(figsize=(fig_width, fig_height))
sns.heatmap(
    expr_z,
    cmap="vlag",
    center=0,
    ax=ax,
    cbar_kws={"label": "Mean expression z-score"},
)
ax.set_xlabel("Marker gene")
ax.set_ylabel(label_key)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "bcell_original_label_marker_heatmap.pdf", bbox_inches="tight")
plt.show()
